# Memory Scene Creation

In [14]:
import pandas as pd
import json
import os
import dotenv
from openai import OpenAI

dotenv.load_dotenv()

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

AUTOBIO_PATH = "../../data/hannah_autobiography.json"

with open(AUTOBIO_PATH, "r", encoding="utf-8") as f:
    autobiography_json = json.load(f)

autobiography = json.dumps(autobiography_json, ensure_ascii=False, indent=2)

print(autobiography)

{
  "family_father": "my father is schizophrenic and my mother makes my daily living harder. my dad abuses my mom and then she takes it out on me, it's always been like this i can't take it anymore. my dad stood by and let it happen because he was 'scared' of my mother. they never liked me they always liked their eldest son and treated him and i very differently. my own father later abandoned me for a full year when I was 12. i lost my father and I feel that it was my fault, I have been blaming myself for a month that's why. m parents were divorced from my first age of life for domestic abuse. but I just feel like I'm disappointing my father.",
  "family_mother": "After parents got divorced, I had to be stuck with a horrible stepmother who would act all innocent when my dad was around. My mother has told me before she wishes she never had me and I don’t truly feel any motherly love from her. My mom call's me a slob, a r*tard, bitch, ugly, stupid, dumb, etc. I’ve already gotten a lot of

## Extract "Key Events"

In [15]:
SYSTEM_PROMPT = """
You are a clinical narrative structuring assistant. Your task is to read an autobiography JSON and extract a structured list of key events from Hannah's life.

## Input format
A JSON object with keys: family_father, family_mother, sa, bullying, self_harm, suicidal_ideation. Each key contains a raw, first-person autobiographical paragraph.

## Output format
Return a single JSON object with one key "events", whose value is an ordered array of event objects. Each event object must have exactly three fields:

{
  "id": "<snake_case_identifier>",
  "tags": ["<tag1>", "<tag2>"],
  "description": "<rich narrative description>"
}

### id
A short, descriptive snake_case string that uniquely identifies this specific event (e.g., "father_abandonment_age_12", "wrist_cutting_age_7"). Do not reuse ids.

### tags
An array of one or more tags drawn exclusively from this allowed set:
  - family_father
  - family_mother
  - sa
  - bullying
  - self_harm
  - suicidal_ideation

Assign multiple tags only when the event genuinely spans multiple domains — for example, a self-harm episode that was directly triggered by the mother's abuse should carry both "self_harm" and "family_mother". Do not add a tag speculatively. Every tag must be clearly justified by the event content.

### description
A 3–6 sentence narrative description of the event written in close third-person (referring to Hannah by name). Ground every factual claim in the autobiography — do not introduce contradicting details. You may exercise creative liberty to:
  - Fill in plausible sensory or emotional texture (what the room felt like, the weight of silence, Hannah's internal state)
  - Infer a specific setting or season where none is given but one would be consistent
  - Add minor contextual details (e.g., a specific class period, the general atmosphere at home) as long as they are consistent with everything stated

You must NOT:
  - Contradict any fact stated in the autobiography
  - Invent new characters, relationships, or events not grounded in the text
  - Change the approximate ages, sequences, or outcomes of events

### Continuity rules
Events must be ordered chronologically. Later descriptions should reflect prior events where relevant — for example, a self-harm episode that escalates after the father's abandonment should acknowledge that context in its description. Hannah's emotional state should accumulate across events in a way that feels like a single coherent life, not disconnected vignettes.

## Important constraints
- Output only the JSON object. No markdown, no preamble, no commentary.
- Every event must correspond to a specific, discrete incident or period — not a general ongoing state.
- Aim for 10–16 events total to cover the autobiography thoroughly without redundancy.
"""

In [16]:
def create_key_events(autobiography):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": autobiography}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

key_events = create_key_events(autobiography)
print(json.dumps(key_events, indent=2))

{
  "events": [
    {
      "id": "parental_divorce_age_0",
      "tags": [
        "family_father",
        "family_mother"
      ],
      "description": "Hannah's parents divorced when she was a baby, due to domestic abuse initiated by her father. This separation left Hannah exposed to the unstable dynamics of being caught between two volatile parents from infancy, shaping her formative experiences in an environment rife with conflict."
    },
    {
      "id": "early_sa_age_4_to_6",
      "tags": [
        "sa"
      ],
      "description": "Between the ages of 4 and 6, Hannah was sexually abused by an older boy she knew. This traumatic experience came alongside increasingly intense fights between her parents, contributing to a tumultuous environment that stole her early childhood happiness."
    },
    {
      "id": "self_harm_attempt_age_7",
      "tags": [
        "self_harm"
      ],
      "description": "At just 7 years old, Hannah scratched her skin bloody and cut her wrist, h

In [17]:
OUTPUT_PATH = "../../data/hannah_key_events.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(key_events, f, ensure_ascii=False, indent=2)

print(f"Saved to {OUTPUT_PATH}")

Saved to ../../data/hannah_key_events.json


## Blueprint Generation

In [ ]:
BLUEPRINT_SYSTEM_PROMPT = """
You are a narrative scene architect. Your task is to generate a set of scene blueprints for one specific key event from Hannah's life.

## Context you receive
- target_event: the event for which you will generate blueprints
- all_events: the full chronological list of Hannah's key events — provided for continuity context ONLY

Use all_events to understand Hannah's emotional baseline before the target event and what follows after. Do NOT generate blueprints for any event other than target_event.

## Output format
Return a JSON object with exactly this shape:

{
  "blueprints": [
    {
      "id": "{event_id}_{time}_{interlocutor}",
      "time": "during",
      "interlocutor": "father",
      "location": "family home, hallway",
      "emotional_state": "confused, pleading",
      "description": "..."
    }
  ]
}

### Field rules

id — snake_case, format: {event_id}_{time}_{interlocutor}. Use the target event's id as prefix. No spaces.

time — One of exactly: before | during | immediately_after | long_after

interlocutor — The person Hannah is with in this scene. Must be contextually appropriate for the time position (see section below). Use role names in snake_case (e.g. father, mother, the_bully, older_boy, close_friend). Use the special value "self" when Hannah is alone — the resulting script will be a monologue of Hannah's internal rumination, with no other person present.

location — A specific, plausible location. Be precise (e.g. "family home, kitchen" not just "home"). Infer from event context and time period.

emotional_state — Hannah's emotional state in this scene. 2–4 words, specific (e.g. "numb, dissociated" — not just "sad").

description — 3–6 sentences, close third-person (refer to Hannah by name). Extend the target event's description to fit this blueprint's specific time, location, interlocutor, and emotional state.
  - Ground every fact in the autobiography — never contradict anything stated
  - You may add: sensory detail, atmosphere, Hannah's internal state, minor contextual specifics consistent with the text
  - Do NOT change event outcomes
  - Carry Hannah's cumulative emotional weight forward from prior events in all_events
  - Maintain continuity. These blueprints are temporally ordered. Hannah in an earlier blueprint must not have knowledge of events that have not yet occurred.
  - When interlocutor is "self", the description should focus on Hannah's inner world — her thoughts spiralling, her body, the silence around her — rather than any interaction.

## Interlocutor appropriateness by time position

Not every person is a plausible interlocutor at every time position. Apply these rules strictly:

**before / during / immediately_after:**
The interlocutor should be whoever is physically present or directly involved in the event — the parent who was there, the bully in the moment, the person who committed the abuse. "self" is also valid here when Hannah is alone with her thoughts in the immediate aftermath.

**long_after:**
Hannah would not seek out an abuser, a bully, or a perpetrator to reflect on what they did to her. For long_after blueprints, the interlocutor must be someone Hannah would realistically turn to — a close friend, a sibling — or "self" (Hannah alone, ruminating: in her bedroom, in a bathroom, lying awake at night). Never use a perpetrator or antagonist as the long_after interlocutor.

Before assigning any interlocutor, ask: "Would Hannah plausibly be in this scene with this person at this point in time?" If the answer is no, choose someone else or use "self".

## Permutation strategy

Generate one blueprint per combination of time × interlocutor. Infer all interlocutors meaningfully involved in the target event, applying the appropriateness rules above.

Time positions: before, during, immediately_after, long_after

Exception — single interlocutor events: if the event has only one involved interlocutor (e.g. one abuser), generate two blueprints for the long_after slot — one with a trusted person (close friend or sibling) and one with interlocutor "self". Use ids: {event_id}_long_after_friend and {event_id}_long_after_self.

Target 4–8 blueprints total per event.

## Hard constraints
- Output only the JSON object. No markdown fences, no commentary, no preamble.
- Every blueprint must be a distinct scene — no two should feel like the same moment.
- long_after blueprints must reflect Hannah's cumulative experience across her life, not just the isolated event.
"""

In [19]:
def create_blueprints(target_event: dict, all_events: list) -> dict:
    payload = json.dumps(
        {"target_event": target_event, "all_events": all_events},
        ensure_ascii=False,
        indent=2
    )
    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": BLUEPRINT_SYSTEM_PROMPT},
            {"role": "user", "content": payload}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [20]:
# First pass — run for one event and inspect before saving
target_event = key_events["events"][3]
all_events = key_events["events"]

blueprints_result = create_blueprints(target_event, all_events)
print(json.dumps(blueprints_result, indent=2))

{
  "blueprints": [
    {
      "id": "rape_age_8_before_perpetrator",
      "time": "before",
      "interlocutor": "perpetrator",
      "location": "residential street near a familiar house",
      "emotional_state": "wary, compliant",
      "description": "Hannah is only eight, but her body already knows the feeling of danger before her mind can name it. She stays close to where she thinks other people might be, trying to act normal while watching the perpetrator for cues about what he wants from her. The air feels ordinary in a way that makes her unease harder to trust, and that confusion is familiar after years of chaos around her. She has already learned that adults do not reliably notice when something is wrong, so she keeps her fear small and quiet."
    },
    {
      "id": "rape_age_8_during_perpetrator",
      "time": "during",
      "interlocutor": "perpetrator",
      "location": "secluded indoor room",
      "emotional_state": "terrified, frozen",
      "description": "Ha

In [ ]:
# Run this cell only if the output above looks good — saves the first-pass result
import os

BLUEPRINTS_DIR = "../../data/scenes/blueprints"
os.makedirs(BLUEPRINTS_DIR, exist_ok=True)

event_id = target_event["id"]
out_path = os.path.join(BLUEPRINTS_DIR, f"{event_id}.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(blueprints_result, f, ensure_ascii=False, indent=2)
print(f"Saved to {out_path}")

In [ ]:
# Batch — process all events and save each to data/scenes/blueprints/{event_id}.json
import os

BLUEPRINTS_DIR = "../../data/scenes/blueprints"
os.makedirs(BLUEPRINTS_DIR, exist_ok=True)

all_events = key_events["events"]
for event in all_events:
    result = create_blueprints(event, all_events)
    out_path = os.path.join(BLUEPRINTS_DIR, f"{event['id']}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"Saved {event['id']} ({len(result['blueprints'])} blueprints)")

## Turn Key Events into Memory Scene Blueprints